# Pre-class: Monday Morning — Session Data + Baseline
**⏱ This pre-class notebook takes approximately 20 minutes (matching the pre-class schedule).**

---

## Scenario: Monday — Marcus's Sequential-Behaviour Question

End of L06 Marcus said:

> *"If we collected DETAILED CUSTOMER BEHAVIOUR DATA — every click, every page view, every cart event — could you predict whether they'd complete checkout WHILE THEY'RE SHOPPING?"*

NorthStar's data team has now logged 8,000 customer sessions. Each session has 9 behavioural features (pages viewed, time on site, items added, etc.) and a binary label (`completed`: did they finish checkout?).

This week Sarah will train three models on this data and compare:
1. **Logistic regression** (L03 baseline)
2. **Gradient boosting** (L04 toolkit)
3. **Neural network** (this week's new tool — MLP in PyTorch)

The point isn't to "win" — gradient boosting probably wins on tabular data this size. The point is to LEARN NEURAL NETWORKS as a foundation for L08 (images), L09 (text), L10 (transformers).

**By the end of this notebook you will be able to:**
- Describe the session-data structure
- Spot which features are most predictive (visually)
- See the L03 logistic-regression baseline accuracy + AUC

In [1]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 4.5)

print("✅ Libraries loaded")

✅ Libraries loaded


## Load the data

The target column **`completed`** is our label: `1` = the customer finished checkout (made a purchase) during that session, `0` = they left without buying. Every other column (except `session_id`) is a behavioural feature we'll use to predict it.

In [ ]:
# Read the session log into a DataFrame (a table of rows × columns).
df = pd.read_csv("data/northstar_sessions.csv")
print(f"Loaded: {len(df):,} sessions × {df.shape[1]} columns")   # rows = sessions, columns = features + id + label
# .mean() of a 0/1 column = the fraction that are 1 → here, the share of sessions that completed checkout.
print(f"Completion rate: {df['completed'].mean():.1%}")
print()
df.head()   # show the first 5 rows so we can eyeball what each column looks like

## Step 1 — Quick EDA

In [ ]:
# .describe() prints count/mean/std/min/quartiles/max for every numeric column —
# a fast way to sanity-check ranges and spot anything odd. Drop the id column (it's just a label, not a feature).
print("Feature summary:")
print(df.drop(columns=["session_id"]).describe().round(2))

## Step 2 — Which features look most predictive?

Plot a bar chart of mean target by feature value (for binary features) and compare distributions by target (for numeric features).

In [ ]:
# Goal: eyeball which features separate "completed" from "not completed".
# A 2×3 grid of small charts: top row = numeric features, bottom row = binary/count features.
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# --- Numeric features: overlay two histograms (completed vs not) to compare their shapes ---
# If the orange and blue humps sit in different places, that feature helps tell the classes apart.
for ax, col in zip(axes.flat[:3], ["pages_viewed", "time_on_site_min", "cart_total_gbp"]):
    df[df["completed"] == 0][col].hist(bins=30, alpha=0.5, ax=ax, color="steelblue", label="Did NOT complete")
    df[df["completed"] == 1][col].hist(bins=30, alpha=0.5, ax=ax, color="coral", label="Completed")
    ax.set_title(col)
    ax.set_xlabel(""); ax.legend(fontsize=9)

# --- Binary / count features: bar of completion RATE per value ---
# groupby(col)["completed"].mean() = "of the sessions with this value, what fraction completed?"
for ax, col in zip(axes.flat[3:], ["items_added", "used_discount_code", "is_returning"]):
    rate_by_val = df.groupby(col)["completed"].mean()
    rate_by_val.plot(kind="bar", ax=ax, color="seagreen", edgecolor="white")
    ax.set_title(f"Completion rate by {col}")
    ax.set_xlabel(col); ax.set_ylabel("Completion rate")
    # dashed line = the overall completion rate, so we can see which bars beat the average
    ax.axhline(df["completed"].mean(), color="gray", linestyle="--", linewidth=1, label="Overall")

plt.tight_layout()
plt.show()

### 💡 What you should notice

- **`items_added`** is dominant — sessions with items in the cart complete much more often than sessions with empty carts.
- **`is_returning`** helps — returning customers convert more (familiarity, trust).
- **`used_discount_code`** is a strong positive signal — people who apply discounts are committed to buying.
- **`time_on_site` and `pages_viewed`** show overlapping distributions — they correlate with completion but not as cleanly.

## Step 3 — Logistic regression baseline (the L03 bar to beat)

In [ ]:
# Split the table into the LABEL (what we predict) and the FEATURES (what we predict from).
y = df["completed"]                                  # 1 = completed checkout, 0 = did not
X = df.drop(columns=["session_id", "completed"])     # drop the id and the answer; keep the 9 behaviour columns

# Hold out 20% as a TEST set; stratify keeps the same completed/not ratio in train and test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,
)

# Every column here is already numeric (binary flags don't strictly need scaling, but it's harmless).
numeric_features = X.columns.tolist()

# A preprocessor that, for the numeric columns: fills any missing values with the median, then rescales
# to mean 0 / std 1 (logistic regression behaves better when features are on a similar scale).
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
])

# A Pipeline chains the preprocessing and the model so .fit() runs both in the right order
# AND learns the scaling on train only — no leakage of test info.
lr_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

lr_pipeline.fit(X_train, y_train)                    # learn the scaler + fit the model in one call
y_pred_lr = lr_pipeline.predict(X_test)              # hard yes/no predictions
y_proba_lr = lr_pipeline.predict_proba(X_test)[:, 1] # probability of class "1" (used for AUC)

print(f"=== Logistic Regression baseline ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")   # fraction of correct yes/no calls
print(f"AUC:      {roc_auc_score(y_test, y_proba_lr):.3f}")   # ranking quality (1.0 = perfect, 0.5 = coin flip)
print()
# precision/recall/F1 broken down per class — shows where the model is strong vs weak
print(classification_report(y_test, y_pred_lr, target_names=["did NOT complete", "completed"]))

### 💡 What this tells us

- **Baseline accuracy ~70%** — better than the 57% **no-skill majority class**. (The "no-skill majority" is the score you'd get by ignoring the features and always guessing the most common answer. Here 57% of sessions did NOT complete, so always predicting "did not complete" would be right 57% of the time. Any useful model must beat that.)
- **AUC ~0.76** — the model can rank sessions but isn't perfectly separating them. (**AUC** = Area Under the ROC Curve. Read it as: pick one completed session and one non-completed session at random — AUC is the probability the model gives the completed one a higher score. 1.0 = perfect ranking, 0.5 = no better than a coin flip.)
- **Logistic regression is LINEAR** — it can only learn linear combinations of features. If there are NONLINEAR interactions (e.g., "high cart total AND new visitor → drop"), it'll miss them.

This is the bar. In class we'll train a Multi-Layer Perceptron and see if its non-linearity helps.

## ✅ Section Summary

| What we did | Result |
|---|---|
| **Loaded session data** | 8,000 sessions × 9 features |
| **Saw the EDA** | items_added is the strongest signal; is_returning + discount also help |
| **Trained L03 logistic regression** | Accuracy ~70%, AUC ~0.76 |

**The bar to beat in class:** can a neural network beat AUC ~0.76? (Spoiler for class: all the models end up tied within ~0.015 AUC — the real prize is learning the PyTorch training loop.)

**Bring to class:**
1. The baseline accuracy + AUC numbers
2. Two features you think will be most important to the neural network
3. One question about how a network "learns" what the features mean

---
**In class → Open `02_perceptron_to_mlp.ipynb` first.**